# Purpose

Prove the workload certification contract authority is bounded, deterministic, and complete.

## Lemma Mapping

- L63 — M365 Workload Certification v1

## Invariant Support

- invariants/lemmas/L63_m365_workload_certification_v1.yaml

## Expected Results

- The workload certification contract validates with four ordered phases.
- Governance rules are complete.
- Per-domain certification verdicts match the routing table.
- Zero-action domains are not-yet-certified.
- Certified plus not-yet-certified counts equal total domain count.

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()
contract_path = repo_root / 'registry' / 'workload_certification_v1.yaml'
contract = yaml.safe_load(contract_path.read_text(encoding='utf-8'))

assert contract['contract']['id'] == 'workload-certification'
assert contract['contract']['status'] == 'contract-defined'

phases = contract['certification_phases']
assert len(phases) == 4
assert [p['order'] for p in phases] == [1, 2, 3, 4]

rules = contract['governance_rules']
assert set(rules.keys()) == {
    'fail_closed', 'audit_completeness', 'bounded_claims',
    'determinism', 'wave_alignment',
}

In [ ]:
routing_path = repo_root / 'registry' / 'executor_routing_v2.yaml'
routing = yaml.safe_load(routing_path.read_text(encoding='utf-8'))

domains = routing['canonical_executor_domains']
routes = routing.get('exact_action_routes', {})

domain_action_counts = {d: 0 for d in domains}
for action, domain in routes.items():
    if domain in domain_action_counts:
        domain_action_counts[domain] += 1

cert_status = contract['domain_certification_status']
kpis = contract['kpis']

assert kpis['total_executor_domains'] == len(domains)
assert len(cert_status) == len(domains)

certified_count = 0
not_yet_count = 0
total_routed = 0

for d in domains:
    entry = cert_status[d]
    assert entry['routed_action_count'] == domain_action_counts[d]
    total_routed += entry['routed_action_count']
    if entry['routed_action_count'] == 0:
        assert entry['certification_verdict'] == 'not-yet-certified'
        not_yet_count += 1
    else:
        assert entry['certification_verdict'] == 'certified'
        certified_count += 1

assert certified_count + not_yet_count == len(domains)
assert kpis['domains_with_routed_actions'] == certified_count
assert kpis['domains_without_routed_actions'] == not_yet_count
assert kpis['total_routed_actions'] == total_routed